# 03 — From opacity to expected muon counts

*Week 5*

This notebook converts the opacity map into expected muon counts. The student-owned `expected_counts` function is stored in `code/student_analysis.py`, so later notebooks do not depend on variables left in memory here.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')


## The conversion

For each angular pixel:

`expected counts = transmitted flux × detector area × pixel solid angle × time`

The transmitted flux comes from `mg.transmitted_flux(opacity, theta)`. The baseline detector area is 0.25 m². The pixel solid angle is approximated by `mg.pixel_solid_angle(step_rad)`.


### YOUR TURN

Implement `expected_counts(...)` in `code/student_analysis.py`. It should:

1. call `opacity_map`,
2. compute the zenith angle for every viewing direction,
3. use `mg.transmitted_flux`,
4. multiply by area, pixel solid angle and exposure time, and
5. return `(counts, ax_deg, ay_deg)`.

Use the baseline parameters in `project_config.py` rather than typing different values into different notebooks.


In [ ]:
# After implementing expected_counts in code/student_analysis.py:
counts_void, ax_deg, ay_deg = sa.expected_counts(
    mg.Pyramid(base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
              density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
              void_radius=cfg.VOID_RADIUS_M, has_void=True),
    np.array(cfg.DETECTOR_POSITION_M, dtype=float),
    days=30,
)
print('30-day count range:', counts_void.min(), counts_void.max())


### YOUR TURN — display it

Show the 30-day expected-count map. Can you see the chamber? The answer should be that the overall pyramid shape dominates. That is why we need a no-chamber reference.


In [ ]:
# YOUR TURN


## Divide out the target shape

Build the same expected-count map for `target.without_void()` and calculate:

`ratio = expected_with_void / expected_without_void`

Away from the chamber the ratio should be close to 1.0. The chamber should appear as an excess above 1.0.


In [ ]:
target = mg.Pyramid(base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
              density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
              void_radius=cfg.VOID_RADIUS_M, has_void=True)
target0 = target.without_void()
detector = np.array(cfg.DETECTOR_POSITION_M, dtype=float)


In [ ]:
# YOUR TURN


### Why this ratio is the reconstruction

The raw count map mostly tells you the shape of the pyramid. The ratio removes that known geometric effect and leaves the chamber contrast. This clean ratio is the “after” image for the before-and-after pair.

Do not treat the clean ratio as noisy detector data; notebook 04 adds Poisson fluctuations.
